# Tiempos totales programados y calibrados de las rutas QroBus

Este notebook calcula la duración completa de cada viaje usando `stop_times.txt`, resume P10, P50, promedio y P90 por ruta y sentido y reconstruye tiempos acumulados desde cada parada. Si existe el histórico local de Google, aprende una calibración robusta `constante + factor × GTFS` para producir escenarios P50 y P80. No realiza llamadas nuevas a Google Maps API.

> Como el GTFS local no contiene `shapes.txt`, el mapa toma el viaje más cercano a la mediana y une sus paradas consecutivas siguiendo la red vial de OpenStreetMap, con los trazos que `01_PromedioRutas.ipynb` deja en `data/processed/segmentos_red_vial.csv`. Es el recorrido más probable por calles, no el derrotero oficial; si ese archivo no existe, las paradas se unen con líneas rectas.


In [1]:
from pathlib import Path
import colorsys
import html
import sys

import folium
import numpy as np
import pandas as pd
from IPython.display import display


def estilizar_tabla(dataframe, titulo):
    """Aplica un estilo visual consistente a las tablas del notebook."""
    estilos = [
        {"selector": "table", "props": [
            ("border-collapse", "separate"), ("border-spacing", "0"),
            ("width", "100%"), ("border", "1px solid #d7e0e8"),
            ("border-radius", "9px"), ("overflow", "hidden"),
            ("font-family", "-apple-system, BlinkMacSystemFont, Segoe UI, sans-serif"),
            ("font-size", "13px"), ("color", "#1f2937"),
        ]},
        {"selector": "caption", "props": [
            ("caption-side", "top"), ("text-align", "left"),
            ("font-size", "15px"), ("font-weight", "700"),
            ("color", "#17324d"), ("padding", "0 0 8px 0"),
        ]},
        {"selector": "thead th", "props": [
            ("background-color", "#17324d"), ("color", "white"),
            ("font-weight", "600"), ("text-align", "left"),
            ("padding", "9px 11px"), ("border-bottom", "3px solid #0f766e"),
            ("white-space", "nowrap"),
        ]},
        {"selector": "tbody td", "props": [
            ("padding", "8px 11px"), ("border-bottom", "1px solid #e8edf2"),
            ("vertical-align", "top"),
        ]},
        {"selector": "tbody th", "props": [
            ("background-color", "#f8fafc"), ("color", "#64748b"),
            ("font-weight", "500"), ("padding", "8px 10px"),
            ("border-bottom", "1px solid #e8edf2"),
        ]},
        {"selector": "tbody tr:nth-child(even)", "props": [("background-color", "#f3f7fa")]},
        {"selector": "tbody tr:hover", "props": [("background-color", "#e9f6f2")]},
    ]
    return (
        dataframe.style
        .set_caption(titulo)
        .set_table_styles(estilos, overwrite=False)
        .format(precision=2, thousands=",", na_rep="—")
        .hide(axis="index")
    )

# Los módulos compartidos viven en src/scripts/; funciona al ejecutar desde
# src/notebooks/, desde src/ o desde la raíz del repositorio.
SCRIPTS_DIR = next(
    base / "scripts"
    for base in [Path.cwd().parent, Path.cwd(), Path.cwd() / "src"]
    if (base / "scripts" / "config.py").exists()
)
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from config import (
    CORREGIDORA_LAT,
    CORREGIDORA_LON,
    FACTOR_DESVIO_CAMINATA,
    GTFS_DIR,
    MAPS_DIR,
    PENALIZACION_TRANSBORDO_MIN,
    PROCESSED_DIR,
    RADIO_BUSQUEDA_PARADA_M,
    VELOCIDAD_CAMINATA_M_MIN,
)
from geo import cargar_trazos, haversine_m
from gtfs_network import gtfs_time_to_seconds, segmentos_medianos_gtfs

RESUMEN_PATH = PROCESSED_DIR / "tiempos_totales_rutas_gtfs.csv"
VIAJES_PATH = PROCESSED_DIR / "duraciones_viajes_gtfs.csv"
OBSERVACIONES_GOOGLE_PATH = PROCESSED_DIR / "google_maps_transit_observaciones.csv"
CALIBRACION_PATH = PROCESSED_DIR / "calibracion_tiempos_rutas_google.csv"
ACUMULADOS_PATH = PROCESSED_DIR / "tiempos_acumulados_paradas_rutas.csv"
MAPA_PATH = MAPS_DIR / "mapa_tiempos_totales_rutas_gtfs.html"
TRAZOS_PATH = PROCESSED_DIR / "segmentos_red_vial.csv"
MIN_OBSERVACIONES_RUTA = 30
MIN_RANGO_GTFS_RUTA_MIN = 10
SEMIVIDA_OBSERVACIONES_DIAS = 14
CONEXIONES_PATH = PROCESSED_DIR / "conexiones_paradas_rutas.csv"
# Una parada a 200 m o menos en línea recta (unos 250 m por calles, ~4 min a
# 4 km/h) se ofrece como conexión caminando. El cálculo de tiempos solo
# transborda en la misma parada. Velocidad y desvío vienen de config.py.
RADIO_CONEXION_A_PIE_M = 200
# Isócronas: bandas acumuladas de tiempo y su color (verde = cerca, rojo = lejos).
UMBRALES_ISOCRONA_MIN = [5, 15, 30, 45, 60]
COLORES_ISOCRONA = {5: "#1a9850", 15: "#91cf60", 30: "#fee08b", 45: "#fc8d59", 60: "#d73027"}
print(f"GTFS: {GTFS_DIR}")
print(f"Mapa: {MAPA_PATH}")


GTFS: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/gtfs
Mapa: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/maps/mapa_tiempos_totales_rutas_gtfs.html


## Funciones del análisis

Las funciones quedan definidas dentro del notebook para que el análisis no dependa de un script externo.


In [2]:
def calcular_duraciones_viajes(stop_times, trips, routes):
    """Devuelve una fila por viaje con su duración programada completa."""
    stop_times = stop_times.copy()
    stop_times["arrival_sec"] = gtfs_time_to_seconds(stop_times["arrival_time"])
    stop_times["departure_sec"] = gtfs_time_to_seconds(stop_times["departure_time"])
    stop_times["stop_sequence"] = pd.to_numeric(
        stop_times["stop_sequence"], errors="raise"
    )
    ordenados = stop_times.sort_values(["trip_id", "stop_sequence"])
    extremos = ordenados.groupby("trip_id", as_index=False).agg(
        stop_id_inicio=("stop_id", "first"),
        stop_id_fin=("stop_id", "last"),
        salida_inicio_sec=("departure_sec", "first"),
        llegada_fin_sec=("arrival_sec", "last"),
        numero_paradas=("stop_id", "size"),
    )
    extremos["duracion_programada_min"] = (
        extremos["llegada_fin_sec"] - extremos["salida_inicio_sec"]
    ) / 60
    invalidos = extremos["duracion_programada_min"].lt(0)
    if invalidos.any():
        ejemplos = extremos.loc[invalidos, "trip_id"].head().tolist()
        raise ValueError(f"Hay viajes con duración negativa: {ejemplos}")
    viajes = (
        extremos.merge(trips, on="trip_id", how="left", validate="one_to_one")
        .merge(routes, on="route_id", how="left", validate="many_to_one")
    )
    if viajes["route_id"].isna().any():
        raise ValueError("Hay trip_id de stop_times que no existen en trips.txt.")
    return viajes


def resumir_por_ruta_sentido(viajes):
    """Resume la variación programada sin ocultarla en un único promedio."""
    def percentil(q):
        return lambda serie: float(np.nanpercentile(serie, q))

    agrupadores = ["route_id", "route_short_name", "route_long_name", "direction_id"]
    resumen = viajes.groupby(
        agrupadores, dropna=False, as_index=False
    ).agg(
        numero_viajes=("trip_id", "nunique"),
        numero_servicios=("service_id", "nunique"),
        duracion_min_min=("duracion_programada_min", "min"),
        duracion_p10_min=("duracion_programada_min", percentil(10)),
        duracion_mediana_min=("duracion_programada_min", "median"),
        duracion_promedio_min=("duracion_programada_min", "mean"),
        duracion_p90_min=("duracion_programada_min", percentil(90)),
        duracion_max_min=("duracion_programada_min", "max"),
        paradas_mediana=("numero_paradas", "median"),
    )
    columnas_tiempo = [c for c in resumen if c.startswith("duracion_")]
    resumen[columnas_tiempo] = resumen[columnas_tiempo].round(2)
    resumen["paradas_mediana"] = resumen["paradas_mediana"].round(1)
    resumen["estado_calculo"] = "calculado"
    return resumen.sort_values(["route_short_name", "direction_id"])


def agregar_rutas_sin_viajes(resumen, routes):
    """Conserva el catálogo completo e identifica rutas sin horarios."""
    route_ids_con_viajes = set(resumen["route_id"].dropna().astype(str))
    faltantes = routes.loc[
        ~routes["route_id"].astype(str).isin(route_ids_con_viajes),
        ["route_id", "route_short_name", "route_long_name"],
    ].copy()
    if faltantes.empty:
        return resumen
    faltantes["direction_id"] = np.nan
    faltantes["numero_viajes"] = 0
    faltantes["numero_servicios"] = 0
    for columna in [
        "duracion_min_min", "duracion_p10_min", "duracion_mediana_min",
        "duracion_promedio_min", "duracion_p90_min", "duracion_max_min",
        "paradas_mediana",
    ]:
        faltantes[columna] = np.nan
    faltantes["estado_calculo"] = "sin viajes en trips.txt"
    return (
        pd.concat([resumen, faltantes[resumen.columns]], ignore_index=True)
        .sort_values(["route_short_name", "direction_id"], na_position="last")
        .reset_index(drop=True)
    )


### Calibración robusta con el histórico de Google

El modelo restringe la constante a valores no negativos y el factor a valores mayores o iguales que uno, porque su objetivo es representar atraso adicional y nunca reducir el horario GTFS. P50 y P80 se ajustan por separado mediante pérdida cuantílica y las observaciones recientes reciben mayor peso.


In [3]:
def percentil_ponderado(valores, pesos, q):
    """Calcula un percentil con pesos no negativos."""
    valores = np.asarray(valores, dtype=float)
    pesos = np.asarray(pesos, dtype=float)
    validos = np.isfinite(valores) & np.isfinite(pesos) & (pesos > 0)
    if not validos.any():
        return np.nan
    valores, pesos = valores[validos], pesos[validos]
    orden = np.argsort(valores)
    valores, pesos = valores[orden], pesos[orden]
    acumulado = np.cumsum(pesos)
    posicion = np.searchsorted(acumulado, q * acumulado[-1], side="left")
    return float(valores[min(posicion, len(valores) - 1)])


def preparar_observaciones_google(observaciones):
    """Limpia el histórico y calcula pesos de recencia sin llamar a la API."""
    requeridas = {
        "observed_at_utc", "route_id_principal", "tiempo_gtfs_min",
        "google_eta_min", "condition", "status_code",
    }
    faltantes = requeridas - set(observaciones.columns)
    if faltantes:
        raise ValueError(f"El histórico de Google no contiene: {sorted(faltantes)}")
    datos = observaciones.copy()
    datos["route_id_principal"] = (
        datos["route_id_principal"].astype("string").str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )
    datos["tiempo_gtfs_min"] = pd.to_numeric(datos["tiempo_gtfs_min"], errors="coerce")
    datos["google_eta_min"] = pd.to_numeric(datos["google_eta_min"], errors="coerce")
    datos["status_code"] = pd.to_numeric(datos["status_code"], errors="coerce")
    datos["observed_at_utc"] = pd.to_datetime(
        datos["observed_at_utc"], errors="coerce", utc=True, format="mixed"
    )
    datos = datos[
        datos["condition"].eq("ROUTE_EXISTS")
        & datos["status_code"].fillna(0).eq(0)
        & datos["tiempo_gtfs_min"].between(1, 180, inclusive="both")
        & datos["google_eta_min"].between(1, 180, inclusive="both")
        & datos["observed_at_utc"].notna()
    ].copy()
    razon = datos["google_eta_min"] / datos["tiempo_gtfs_min"]
    datos = datos[razon.between(0.25, 4, inclusive="both")].copy()
    if datos.empty:
        raise ValueError("No existen observaciones válidas para calibrar tiempos.")
    referencia = datos["observed_at_utc"].max()
    edad_dias = (referencia - datos["observed_at_utc"]).dt.total_seconds() / 86400
    datos["peso_recencia"] = 0.5 ** (edad_dias / SEMIVIDA_OBSERVACIONES_DIAS)
    return datos


def ajustar_modelo_cuantil(datos, q):
    """Ajusta Google = constante + factor × GTFS con restricciones de atraso."""
    x = datos["tiempo_gtfs_min"].to_numpy(dtype=float)
    y = datos["google_eta_min"].to_numpy(dtype=float)
    pesos = datos["peso_recencia"].to_numpy(dtype=float)
    limite_inferior, limite_superior = 1.0, 2.5
    mejor = None
    for _ in range(3):
        factores = np.linspace(limite_inferior, limite_superior, 301)
        for factor in factores:
            constante = max(0.0, percentil_ponderado(y - factor * x, pesos, q))
            error = y - (constante + factor * x)
            perdida = np.sum(
                pesos * np.where(error >= 0, q * error, (q - 1) * error)
            )
            if mejor is None or perdida < mejor[0]:
                mejor = (float(perdida), float(constante), float(factor))
        paso = (limite_superior - limite_inferior) / 300
        limite_inferior = max(1.0, mejor[2] - 2 * paso)
        limite_superior = min(2.5, mejor[2] + 2 * paso)
    return mejor[1], mejor[2]


def calibrar_modelos_por_ruta(observaciones):
    """Ajusta modelos por ruta y los contrae hacia un modelo global."""
    global_p50 = ajustar_modelo_cuantil(observaciones, 0.50)
    global_p80 = ajustar_modelo_cuantil(observaciones, 0.80)
    registros = [{
        "route_id": "__GLOBAL__",
        "observaciones_google": int(len(observaciones)),
        "rango_gtfs_observado_min": float(
            observaciones["tiempo_gtfs_min"].max()
            - observaciones["tiempo_gtfs_min"].min()
        ),
        "peso_modelo_ruta": 0.0,
        "constante_p50_min": global_p50[0],
        "factor_p50": global_p50[1],
        "constante_p80_min": max(global_p80[0], global_p50[0]),
        "factor_p80": max(global_p80[1], global_p50[1]),
        "fuente_modelo": "global",
    }]
    for route_id, grupo in observaciones.dropna(
        subset=["route_id_principal"]
    ).groupby("route_id_principal"):
        rango = float(grupo["tiempo_gtfs_min"].max() - grupo["tiempo_gtfs_min"].min())
        if len(grupo) < MIN_OBSERVACIONES_RUTA or rango < MIN_RANGO_GTFS_RUTA_MIN:
            continue
        ruta_p50 = ajustar_modelo_cuantil(grupo, 0.50)
        ruta_p80 = ajustar_modelo_cuantil(grupo, 0.80)
        peso = (len(grupo) / (len(grupo) + 100)) * min(1.0, rango / 20)
        constante_p50 = (1 - peso) * global_p50[0] + peso * ruta_p50[0]
        factor_p50 = (1 - peso) * global_p50[1] + peso * ruta_p50[1]
        constante_p80 = (1 - peso) * global_p80[0] + peso * ruta_p80[0]
        factor_p80 = (1 - peso) * global_p80[1] + peso * ruta_p80[1]
        registros.append({
            "route_id": str(route_id),
            "observaciones_google": int(len(grupo)),
            "rango_gtfs_observado_min": rango,
            "peso_modelo_ruta": float(peso),
            "constante_p50_min": float(constante_p50),
            "factor_p50": float(factor_p50),
            "constante_p80_min": float(max(constante_p80, constante_p50)),
            "factor_p80": float(max(factor_p80, factor_p50)),
            "fuente_modelo": "ruta suavizada con global",
        })
    return pd.DataFrame(registros)


In [4]:
def seleccionar_viajes_representativos(viajes, resumen):
    """Elige por ruta y sentido el viaje más cercano a la duración mediana."""
    calculados = resumen.loc[resumen["estado_calculo"].eq("calculado")].copy()
    calculados["direction_key"] = (
        pd.to_numeric(calculados["direction_id"], errors="coerce")
        .astype("Int64").astype("string").fillna("sin_sentido")
    )
    viajes_candidatos = viajes.copy()
    viajes_candidatos["direction_key"] = (
        pd.to_numeric(viajes_candidatos["direction_id"], errors="coerce")
        .astype("Int64").astype("string").fillna("sin_sentido")
    )
    columnas_resumen = [
        "route_id", "direction_key", "route_short_name", "route_long_name",
        "numero_viajes", "duracion_p10_min", "duracion_mediana_min",
        "duracion_p90_min", "constante_p50_min", "factor_p50",
        "constante_p80_min", "factor_p80",
        "tiempo_pasajero_p50_min", "tiempo_pasajero_p80_min",
        "observaciones_google", "fuente_modelo",
    ]
    candidatos = viajes_candidatos.merge(
        calculados[columnas_resumen],
        on=["route_id", "direction_key"], how="inner",
        validate="many_to_one", suffixes=("", "_resumen"),
    )
    candidatos["distancia_mediana_min"] = (
        candidatos["duracion_programada_min"]
        - candidatos["duracion_mediana_min"]
    ).abs()
    return (
        candidatos.sort_values(
            ["route_id", "direction_key", "distancia_mediana_min", "trip_id"]
        )
        .drop_duplicates(["route_id", "direction_key"], keep="first")
        .reset_index(drop=True)
    )


def trazar_secuencia_paradas(grupo, trazos_viales):
    """Une las paradas de un viaje por calles; devuelve (coordenadas, km)."""
    ids = grupo["stop_id"].astype(str).tolist()
    puntos = grupo[["stop_lat", "stop_lon"]].astype(float).to_numpy().tolist()
    coordenadas, metros = [puntos[0]], 0.0
    for stop_a, stop_b, punto_a, punto_b in zip(ids, ids[1:], puntos, puntos[1:]):
        trazo = trazos_viales.get((stop_a, stop_b))
        if trazo is None:
            coordenadas.append(punto_b)
            metros += float(haversine_m(*punto_a, *punto_b))
        else:
            coordenadas.extend(trazo["coordenadas"][1:])
            metros += trazo["distancia_red_m"]
    return coordenadas, metros / 1000


def construir_trazos_representativos(
    stop_times, stops, representativos, trazos_viales=None
):
    """Construye los trazos por calles de los viajes representativos."""
    trazos_viales = trazos_viales or {}
    seleccion = representativos[
        [
            "trip_id", "route_id", "direction_key",
            "route_short_name_resumen", "route_long_name_resumen",
            "numero_viajes", "duracion_p10_min",
            "duracion_mediana_min", "duracion_p90_min",
            "constante_p50_min", "factor_p50",
            "constante_p80_min", "factor_p80",
            "tiempo_pasajero_p50_min", "tiempo_pasajero_p80_min",
            "observaciones_google", "fuente_modelo",
        ]
    ].copy()
    puntos = stop_times.loc[
        stop_times["trip_id"].isin(set(seleccion["trip_id"])),
        ["trip_id", "stop_id", "stop_sequence"],
    ].copy()
    puntos["stop_sequence"] = pd.to_numeric(
        puntos["stop_sequence"], errors="raise"
    )
    puntos = (
        puntos.merge(
            stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]],
            on="stop_id", how="left", validate="many_to_one",
        )
        .merge(seleccion, on="trip_id", how="left", validate="many_to_one")
        .dropna(subset=["stop_lat", "stop_lon"])
        .sort_values(["trip_id", "stop_sequence"])
    )
    trazos = []
    for trip_id, grupo in puntos.groupby("trip_id", sort=False):
        primera = grupo.iloc[0]
        if len(grupo) < 2:
            continue
        coordenadas, distancia_km = trazar_secuencia_paradas(grupo, trazos_viales)
        trazos.append(
            {
                "trip_id": str(trip_id),
                "route_id": str(primera["route_id"]),
                "direction_key": str(primera["direction_key"]),
                "route_short_name": str(primera["route_short_name_resumen"]),
                "route_long_name": str(primera["route_long_name_resumen"]),
                "numero_viajes": int(primera["numero_viajes"]),
                "p10_min": float(primera["duracion_p10_min"]),
                "p50_min": float(primera["duracion_mediana_min"]),
                "p90_min": float(primera["duracion_p90_min"]),
                "estimado_p50_min": float(primera["tiempo_pasajero_p50_min"]),
                "estimado_p80_min": float(primera["tiempo_pasajero_p80_min"]),
                "observaciones_google": int(primera["observaciones_google"]),
                "fuente_modelo": str(primera["fuente_modelo"]),
                "distancia_km": distancia_km,
                "coordenadas": coordenadas,
            }
        )
    return trazos


In [5]:
def construir_tiempos_acumulados(stop_times, stops, representativos):
    """Propaga segmentos corregidos hacia atrás desde el final de cada viaje."""
    columnas_modelo = [
        "trip_id", "route_id", "direction_key",
        "route_short_name_resumen", "constante_p50_min", "factor_p50",
        "constante_p80_min", "factor_p80", "observaciones_google",
        "fuente_modelo",
    ]
    seleccion = representativos[columnas_modelo].copy()
    puntos = stop_times.loc[
        stop_times["trip_id"].isin(set(seleccion["trip_id"])),
        ["trip_id", "stop_id", "stop_sequence", "arrival_time", "departure_time"],
    ].copy()
    puntos["stop_sequence"] = pd.to_numeric(puntos["stop_sequence"], errors="raise")
    puntos["arrival_sec"] = gtfs_time_to_seconds(puntos["arrival_time"])
    puntos["departure_sec"] = gtfs_time_to_seconds(puntos["departure_time"])
    puntos = (
        puntos.merge(seleccion, on="trip_id", how="inner", validate="many_to_one")
        .merge(stops[["stop_id", "stop_name"]], on="stop_id", how="left", validate="many_to_one")
        .sort_values(["trip_id", "stop_sequence"])
    )
    resultados = []
    for _, grupo in puntos.groupby("trip_id", sort=False):
        grupo = grupo.copy().reset_index(drop=True)
        siguiente_llegada = grupo["arrival_sec"].shift(-1)
        segmento_gtfs = ((siguiente_llegada - grupo["departure_sec"]) / 60).fillna(0)
        if not segmento_gtfs.iloc[:-1].between(0, 180, inclusive="both").all():
            raise ValueError(f"Segmentos inválidos en el viaje {grupo.iloc[0]['trip_id']}")
        factor_p50 = float(grupo.iloc[0]["factor_p50"])
        factor_p80 = float(grupo.iloc[0]["factor_p80"])
        segmento_p50 = segmento_gtfs * factor_p50
        segmento_p80 = segmento_gtfs * factor_p80
        restante_gtfs = segmento_gtfs.iloc[::-1].cumsum().iloc[::-1]
        restante_p50 = segmento_p50.iloc[::-1].cumsum().iloc[::-1]
        restante_p80 = segmento_p80.iloc[::-1].cumsum().iloc[::-1]
        hay_recorrido = segmento_gtfs.index < len(grupo) - 1
        grupo["next_stop_id"] = grupo["stop_id"].shift(-1)
        grupo["tiempo_segmento_gtfs_min"] = segmento_gtfs
        grupo["tiempo_segmento_p50_min"] = segmento_p50
        grupo["tiempo_segmento_p80_min"] = segmento_p80
        grupo["tiempo_restante_gtfs_min"] = restante_gtfs
        grupo["tiempo_restante_operativo_p50_min"] = restante_p50
        grupo["tiempo_restante_operativo_p80_min"] = restante_p80
        grupo["tiempo_restante_pasajero_p50_min"] = restante_p50 + np.where(
            hay_recorrido, float(grupo.iloc[0]["constante_p50_min"]), 0.0
        )
        grupo["tiempo_restante_pasajero_p80_min"] = restante_p80 + np.where(
            hay_recorrido, float(grupo.iloc[0]["constante_p80_min"]), 0.0
        )
        resultados.append(grupo)
    columnas = [
        "route_id", "route_short_name_resumen", "direction_key", "trip_id",
        "stop_sequence", "stop_id", "stop_name", "next_stop_id",
        "tiempo_segmento_gtfs_min", "tiempo_segmento_p50_min",
        "tiempo_segmento_p80_min", "tiempo_restante_gtfs_min",
        "tiempo_restante_operativo_p50_min",
        "tiempo_restante_operativo_p80_min",
        "tiempo_restante_pasajero_p50_min",
        "tiempo_restante_pasajero_p80_min",
        "constante_p50_min", "factor_p50", "constante_p80_min",
        "factor_p80", "observaciones_google", "fuente_modelo",
    ]
    return pd.concat(resultados, ignore_index=True)[columnas]


### Conexiones entre rutas

Una **conexión en la misma parada** existe cuando dos o más rutas distintas pasan por el mismo `stop_id`; es el único transbordo que usa el Dijkstra de `01` y `02`. Una **conexión a pie** es otra parada a `RADIO_CONEXION_A_PIE_M` metros o menos en línea recta que ofrece alguna ruta que no pasa por la parada de origen. Esta segunda es una referencia para el usuario y no modifica ningún tiempo calculado.

Se usan las paradas de los viajes representativos, es decir, las mismas que forman las líneas del mapa.

In [6]:
def calcular_conexiones(stop_times, stops, representativos, radio_caminata_m):
    """Identifica dónde se cruzan las rutas dibujadas y a qué paradas conectan.

    - Misma parada: dos o más rutas distintas pasan por el mismo stop_id. Es el
      único transbordo que considera el Dijkstra de 01 y 02.
    - A pie: otra parada a no más de `radio_caminata_m` en línea recta que
      ofrece al menos una ruta que no pasa por la parada de origen. Es una
      referencia para el usuario; el cálculo de tiempos no la utiliza.

    Devuelve (servicio, conexiones, paradas_por_trip):
    servicio tiene una fila por parada y ruta-sentido; conexiones, una fila por
    parada de origen y parada conectada; paradas_por_trip, la secuencia de
    paradas de cada viaje representativo.
    """
    seleccion = representativos[
        ["trip_id", "route_id", "direction_key", "route_short_name_resumen"]
    ].rename(columns={"route_short_name_resumen": "route_short_name"})
    puntos = stop_times.loc[
        stop_times["trip_id"].isin(set(seleccion["trip_id"])),
        ["trip_id", "stop_id", "stop_sequence"],
    ].copy()
    puntos["stop_sequence"] = pd.to_numeric(puntos["stop_sequence"], errors="raise")
    puntos = puntos.sort_values(["trip_id", "stop_sequence"])
    paradas_por_trip = {
        str(trip_id): grupo["stop_id"].astype(str).tolist()
        for trip_id, grupo in puntos.groupby("trip_id", sort=False)
    }
    servicio = (
        puntos.merge(seleccion, on="trip_id", how="inner", validate="many_to_one")
        .merge(
            stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]],
            on="stop_id", how="inner", validate="many_to_one",
        )
        .drop_duplicates(["stop_id", "route_id", "direction_key"])
        [["stop_id", "stop_name", "stop_lat", "stop_lon",
          "route_id", "route_short_name", "direction_key"]]
        .reset_index(drop=True)
    )
    rutas_por_parada = servicio.groupby("stop_id")["route_id"].agg(
        lambda rutas: frozenset(rutas.astype(str))
    )
    nombres_ruta = (
        servicio.drop_duplicates("route_id")
        .set_index("route_id")["route_short_name"].astype(str).to_dict()
    )
    paradas = (
        servicio.drop_duplicates("stop_id")
        .set_index("stop_id")[["stop_name", "stop_lat", "stop_lon"]]
    )

    def texto_rutas(route_ids):
        return ", ".join(sorted(nombres_ruta.get(r, r) for r in route_ids))

    registros = []
    for stop_id, rutas in rutas_por_parada.items():
        if len(rutas) >= 2:
            registros.append({
                "stop_id": stop_id,
                "tipo_conexion": "Misma parada",
                "stop_id_conexion": stop_id,
                "distancia_m": 0.0,
                "route_ids_conexion": sorted(rutas),
            })

    # Distancias en bloques para no crear de golpe la matriz completa.
    ids = paradas.index.to_numpy()
    lat = paradas["stop_lat"].to_numpy(dtype=float)
    lon = paradas["stop_lon"].to_numpy(dtype=float)
    for inicio in range(0, len(ids), 400):
        fin = min(inicio + 400, len(ids))
        distancias = haversine_m(
            lat[inicio:fin, None], lon[inicio:fin, None], lat[None, :], lon[None, :]
        )
        filas, columnas = np.nonzero(distancias <= radio_caminata_m)
        for fila, columna in zip(filas, columnas):
            origen, destino = ids[inicio + fila], ids[columna]
            if origen == destino:
                continue
            rutas_nuevas = rutas_por_parada[destino] - rutas_por_parada[origen]
            if rutas_nuevas:
                registros.append({
                    "stop_id": origen,
                    "tipo_conexion": "A pie",
                    "stop_id_conexion": destino,
                    "distancia_m": float(distancias[fila, columna]),
                    "route_ids_conexion": sorted(rutas_nuevas),
                })

    conexiones = pd.DataFrame(registros, columns=[
        "stop_id", "tipo_conexion", "stop_id_conexion",
        "distancia_m", "route_ids_conexion",
    ])
    conexiones = conexiones.merge(
        paradas, left_on="stop_id", right_index=True, how="left"
    ).merge(
        paradas[["stop_name"]].rename(columns={"stop_name": "stop_name_conexion"}),
        left_on="stop_id_conexion", right_index=True, how="left",
    )
    conexiones["rutas_en_parada"] = conexiones["stop_id"].map(
        lambda s: texto_rutas(rutas_por_parada[s])
    )
    conexiones["rutas_conexion"] = conexiones["route_ids_conexion"].map(texto_rutas)
    # Minutos por calles: línea recta × factor de desvío, a la velocidad de caminata.
    conexiones["minutos_caminando"] = (
        conexiones["distancia_m"] * FACTOR_DESVIO_CAMINATA / VELOCIDAD_CAMINATA_M_MIN
    )
    conexiones = conexiones.sort_values(
        ["stop_id", "tipo_conexion", "distancia_m"], ascending=[True, False, True]
    ).reset_index(drop=True)
    return servicio, conexiones, paradas_por_trip


In [7]:
import json

from branca.element import MacroElement
from jinja2 import Template


class ScriptMapa(MacroElement):
    """Inserta JavaScript propio después de las capas y controles del mapa."""

    _template = Template("{% macro script(this, kwargs) %}{{ this.codigo }}{% endmacro %}")

    def __init__(self, codigo):
        super().__init__()
        self._name = "ScriptMapa"
        self.codigo = codigo


def json_para_script(datos):
    """Serializa datos para incrustarlos de forma segura dentro de <script>."""
    return json.dumps(datos, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")


def colores_por_ruta(route_ids):
    """Asigna a cada route_id un tono distinto con la razón áurea."""
    colores = {}
    for indice, route_id in enumerate(sorted({str(r) for r in route_ids})):
        tono = (indice * 0.618033988749895) % 1.0
        rojo, verde, azul = colorsys.hsv_to_rgb(tono, 0.72, 0.82)
        colores[route_id] = (
            f"#{round(rojo * 255):02x}{round(verde * 255):02x}{round(azul * 255):02x}"
        )
    return colores


# Los marcadores y popups de conexiones se generan en el navegador a partir de
# un solo bloque de datos; así el HTML no repite el mismo contenido miles de veces.
JS_CONEXIONES = """
(function () {
  const D = __DATOS__;
  const capaMisma = __CAPA_MISMA__;
  const capaPie = __CAPA_PIE__;
  const lineas = __LINEAS__;
  const esc = (t) => String(t).replace(/[&<>"']/g, (c) => ({
    "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;"
  })[c]);
  const chip = (r, sentido) => {
    const info = D.rutas[r] || [r, "#6b7280"];
    const s = sentido == null ? "" : " · S" + esc(sentido);
    return "<span style='display:inline-block;margin:1px 2px;padding:0 4px;" +
      "border-left:8px solid " + info[1] + ";background:#f3f4f6;border-radius:2px;" +
      "white-space:nowrap'>" + esc(info[0]) + s + "</span>";
  };
  const rutasDe = (s) => new Set(D.paradas[s][3].map((x) => x[0]));
  const caja = (contenido) => "<div style='font-size:12px;max-height:280px;" +
    "overflow-y:auto;min-width:240px'>" + contenido + "</div>";

  function popupParada(s) {
    const p = D.paradas[s];
    const rutas = p[3].slice().sort((a, b) =>
      (D.rutas[a[0]] || [a[0]])[0].localeCompare((D.rutas[b[0]] || [b[0]])[0]) ||
      String(a[1]).localeCompare(String(b[1])));
    let h = "<b>" + esc(p[0]) + "</b> <span style='color:#6b7280'>(parada " +
      esc(s) + ")</span><div style='margin-top:4px'><b>Rutas en esta parada (" +
      rutasDe(s).size + "):</b><br>" + rutas.map((x) => chip(x[0], x[1])).join("") + "</div>";
    const pie = D.apie[s] || [];
    if (pie.length) {
      const max = 12;
      const filas = pie.slice(0, max).map((c) => "<li style='margin:3px 0'>" +
        esc(D.paradas[c[0]][0]) + " <span style='color:#6b7280'>(" + esc(c[0]) + ", " +
        c[1] + " m · ~" + Math.max(1, Math.round(c[1] * D.desvio / D.velocidad)) + " min)</span><br>" +
        c[2].map((r) => chip(r)).join("") + "</li>");
      if (pie.length > max) {
        filas.push("<li style='color:#6b7280'>y " + (pie.length - max) + " paradas más</li>");
      }
      h += "<div style='margin-top:6px'><b>Conexiones a pie (≤ " + D.radio +
        " m en línea recta):</b><ul style='margin:2px 0;padding-left:16px'>" +
        filas.join("") + "</ul></div>";
    }
    h += "<button type='button' data-iso-origen='" + esc(s) + "' style='margin-top:7px;" +
      "padding:3px 7px;cursor:pointer'>Isócrona desde esta parada</button>";
    return caja(h);
  }

  function popupRuta(trip, propia, encabezado) {
    const filas = [];
    (D.trips[trip] || []).forEach((s) => {
      const otras = [...rutasDe(s)].filter((r) => r !== propia);
      const vistas = new Set([propia, ...otras]);
      const pie = [];
      (D.apie[s] || []).forEach((c) => c[2].forEach((r) => {
        if (!vistas.has(r)) { vistas.add(r); pie.push(r); }
      }));
      if (!otras.length && !pie.length) return;
      let detalle = otras.map((r) => chip(r)).join("");
      if (pie.length) {
        detalle += "<span style='color:#6b7280'> a pie: </span>" + pie.map((r) => chip(r)).join("");
      }
      filas.push("<li style='margin:3px 0'>" + esc(D.paradas[s][0]) + "<br>" + detalle + "</li>");
    });
    const lista = filas.length
      ? "<ul style='margin:2px 0;padding-left:16px'>" + filas.join("") + "</ul>"
      : "<div style='color:#6b7280'>Sin conexiones con otras rutas.</div>";
    return caja(encabezado + "<div style='margin-top:6px'><b>Paradas con conexión (" +
      filas.length + "):</b>" + lista + "</div>");
  }

  // El tamaño crece con el número de rutas y se ajusta al zoom para no tapar las líneas.
  const escala = (z) => (z <= 11 ? 0.45 : z <= 12 ? 0.65 : z <= 13 ? 0.85 : 1.1);
  const marcadores = [];
  D.misma.forEach((s) => {
    const p = D.paradas[s];
    const rutas = rutasDe(s);
    const nombres = [...rutas].map((r) => (D.rutas[r] || [r])[0]).sort().join(", ");
    const base = 3 + Math.min(rutas.size, 12) * 0.45;
    const m = L.circleMarker([p[1], p[2]], {
      radius: base, color: "#111827", weight: 1.5,
      fill: true, fillColor: "#ffffff", fillOpacity: 0.95
    }).bindTooltip(esc(p[0]) + " · " + rutas.size + " rutas: " + esc(nombres))
      .bindPopup(() => popupParada(s), { maxWidth: 360 })
      .addTo(capaMisma);
    marcadores.push([m, base]);
  });
  D.pares.forEach((par) => {
    const a = D.paradas[par[0]], b = D.paradas[par[1]];
    L.polyline([[a[1], a[2]], [b[1], b[2]]], {
      color: "#374151", weight: 2, opacity: 0.8, dashArray: "4 4"
    }).bindTooltip(esc(a[0]) + " ↔ " + esc(b[0]) + " · " + par[2] + " m (~" +
      Math.max(1, Math.round(par[2] * D.desvio / D.velocidad)) + " min)")
      .addTo(capaPie);
  });
  D.solo_pie.forEach((s) => {
    const p = D.paradas[s];
    const m = L.circleMarker([p[1], p[2]], {
      radius: 3, color: "#6b7280", weight: 1, fill: true,
      fillColor: "#ffffff", fillOpacity: 0.95
    }).bindTooltip(esc(p[0]) + " · conexión a pie")
      .bindPopup(() => popupParada(s), { maxWidth: 360 })
      .addTo(capaPie);
    marcadores.push([m, 3]);
  });
  const mapa = __MAPA__;
  const ajustar = () => {
    const f = escala(mapa.getZoom());
    marcadores.forEach(([m, base]) => m.setRadius(base * f));
  };
  mapa.on("zoomend", ajustar);
  ajustar();
  lineas.forEach((l) => l[0].bindPopup(() => popupRuta(l[1], l[2], l[3]), { maxWidth: 360 }));
})();
"""


# Isócronas: el navegador recalcula el alcance cada vez que cambia el origen.
JS_ISOCRONAS = """
(function () {
  const I = __ISO__;
  const mapa = __MAPA__;
  const N = I.paradas.length, R = I.rutas.length;
  // Estados por parada: 0 = llegó caminando desde otra parada, 1 = acceso desde
  // el origen, 2 + r = a bordo de la ruta r.
  const SL = R + 2;
  const T_MAX = I.umbrales[I.umbrales.length - 1];
  const esc = (t) => String(t).replace(/[&<>"']/g, (c) => ({
    "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;"
  })[c]);
  const RAD = Math.PI / 180;
  const distM = (la1, lo1, la2, lo2) => {
    const a = Math.sin((la2 - la1) * RAD / 2) ** 2 +
      Math.cos(la1 * RAD) * Math.cos(la2 * RAD) * Math.sin((lo2 - lo1) * RAD / 2) ** 2;
    return 6371008.8 * 2 * Math.atan2(Math.sqrt(a), Math.sqrt(1 - a));
  };
  const minCaminando = (m) => m * I.desvio / I.velocidad;
  const radioCaminando = (min) => Math.min(min * I.velocidad / I.desvio, I.radio);
  const banda = (t) => I.umbrales.find((u) => t <= u);
  const indice = Object.fromEntries(I.paradas.map((p, i) => [p[0], i]));
  const salida = Array.from({ length: N }, () => []);
  I.tramos.forEach((t, k) => salida[t[0]].push(k));
  const vecinosPie = Array.from({ length: N }, () => []);
  I.pie.forEach((p) => { vecinosPie[p[0]].push([p[1], p[2]]); vecinosPie[p[1]].push([p[0], p[2]]); });

  // Cada banda va en su panel: dentro de un panel los círculos se funden en una
  // sola mancha y la transparencia se aplica al panel completo.
  const render = {};
  I.umbrales.forEach((u, k) => {
    const panel = mapa.createPane("iso" + u);
    panel.style.zIndex = 360 - k;
    panel.style.opacity = 0.3;
    panel.style.pointerEvents = "none";
    render[u] = L.canvas({ pane: "iso" + u });
  });
  mapa.createPane("isoLineas").style.zIndex = 380;
  mapa.createPane("isoParadas").style.zIndex = 390;
  const renderLineas = L.canvas({ pane: "isoLineas" });
  const renderParadas = L.canvas({ pane: "isoParadas" });
  const capaIso = L.layerGroup().addTo(mapa);
  const capaParadas = L.layerGroup().addTo(mapa);

  const panel = document.getElementById("panel-isocronas");
  L.DomEvent.disableClickPropagation(panel);
  L.DomEvent.disableScrollPropagation(panel);
  const etiquetaOrigen = document.getElementById("iso-origen");
  const escenario = document.getElementById("iso-escenario");
  const penalizacion = document.getElementById("iso-penalizacion");
  const usarPie = document.getElementById("iso-pie");
  const cuerpoTabla = document.getElementById("iso-tabla");
  const buscador = document.getElementById("iso-buscar");
  const lista = document.getElementById("iso-paradas");
  const filtro = document.getElementById("iso-filtro");
  const acumulado = document.getElementById("iso-acumulado");
  const tituloLista = document.getElementById("iso-lista-titulo");
  const cuerpoLista = document.getElementById("iso-lista");
  I.umbrales.forEach((u) => {
    const opcion = document.createElement("option");
    opcion.value = u;
    filtro.appendChild(opcion);
  });
  const desdeBanda = (u) => { const k = I.umbrales.indexOf(u); return k ? I.umbrales[k - 1] : 0; };
  let lineas = {};       // polilíneas del árbol de caminos, agrupadas por banda
  let visibles = [];     // paradas que pasan el filtro, ordenadas por minutos
  const porOpcion = new Map();
  I.paradas.forEach((p, i) => {
    const texto = p[1] + " [" + p[0] + "]";
    porOpcion.set(texto, i);
    const opcion = document.createElement("option");
    opcion.value = texto;
    lista.appendChild(opcion);
  });

  const llegada = new Float64Array(N).fill(Infinity);
  const marcadores = I.paradas.map((p, i) => L.circleMarker([p[2], p[3]], {
    renderer: renderParadas, radius: 2, weight: 0.5, color: "#4b5563",
    fillColor: "#d1d5db", fillOpacity: 0.9
  }).bindTooltip(() => esc(p[1]) + (Number.isFinite(llegada[i])
      ? " · " + llegada[i].toFixed(1) + " min desde el origen"
      : " · más de " + T_MAX + " min") + "<br><i>Clic: usar como origen</i>")
    .on("click", () => elegirParada(i))
    .addTo(capaParadas));
  const estacion = L.latLng(I.estacion[0], I.estacion[1]);
  L.circleMarker(estacion, {
    radius: 9, color: "#7c2d12", weight: 2, fillColor: "#fb923c", fillOpacity: 1,
    pane: "isoParadas"
  }).bindTooltip("Estación Corregidora · clic: usar como origen")
    .on("click", () => elegirEstacion()).addTo(mapa);
  let marcadorOrigen = null;
  let origen = null;

  // Montículo binario mínimo para Dijkstra.
  function crearMonticulo() {
    const costos = [], estados = [];
    return {
      get size() { return costos.length; },
      push(c, e) {
        let i = costos.length; costos.push(c); estados.push(e);
        while (i > 0) {
          const p = (i - 1) >> 1;
          if (costos[p] <= c) break;
          costos[i] = costos[p]; estados[i] = estados[p]; i = p;
        }
        costos[i] = c; estados[i] = e;
      },
      pop() {
        const c0 = costos[0], e0 = estados[0];
        const c = costos.pop(), e = estados.pop();
        if (costos.length) {
          let i = 0;
          while (true) {
            let h = 2 * i + 1;
            if (h >= costos.length) break;
            if (h + 1 < costos.length && costos[h + 1] < costos[h]) h++;
            if (costos[h] >= c) break;
            costos[i] = costos[h]; estados[i] = estados[h]; i = h;
          }
          costos[i] = c; estados[i] = e;
        }
        return [c0, e0];
      }
    };
  }

  function calcular() {
    const modo = escenario.value;
    const pen = Math.max(0, Number(penalizacion.value) || 0);
    const pie = usarPie.checked;
    const constante = I.rutas.map((r) => modo === "gtfs" ? 0 : modo === "p50" ? r[2] : r[4]);
    const factor = I.rutas.map((r) => modo === "gtfs" ? 1 : modo === "p50" ? r[3] : r[5]);
    const dist = new Float64Array(N * SL).fill(Infinity);
    const previo = new Int32Array(N * SL).fill(-1);
    const via = new Int32Array(N * SL).fill(-1);  // tramo usado; -1 acceso, -2 caminata
    const cola = crearMonticulo();
    const relajar = (estado, costo, desde, tramo) => {
      if (costo <= T_MAX && costo < dist[estado]) {
        dist[estado] = costo; previo[estado] = desde; via[estado] = tramo;
        cola.push(costo, estado);
      }
    };
    I.paradas.forEach((p, j) => {
      const m = distM(origen.lat, origen.lng, p[2], p[3]);
      if (m <= I.radio) relajar(j * SL + 1, minCaminando(m), -1, -1);
    });
    while (cola.size) {
      const [costo, estado] = cola.pop();
      if (costo > dist[estado]) continue;
      const i = Math.floor(estado / SL), slot = estado % SL;
      for (const k of salida[i]) {
        const t = I.tramos[k], r = t[2];
        // Primer abordaje: constante de espera del escenario. Cambio de ruta: transbordo.
        const extra = slot === 1 ? constante[r] : slot === 0 ? pen : (slot - 2 === r ? 0 : pen);
        relajar(t[1] * SL + 2 + r, costo + extra + t[3] * factor[r], estado, k);
      }
      if (pie && slot >= 2) {
        for (const [j, m] of vecinosPie[i]) relajar(j * SL, costo + minCaminando(m), estado, -2);
      }
    }
    const mejor = new Int32Array(N).fill(-1);
    for (let i = 0; i < N; i++) {
      let v = Infinity, b = -1;
      for (let s = 0; s < SL; s++) {
        if (dist[i * SL + s] < v) { v = dist[i * SL + s]; b = i * SL + s; }
      }
      llegada[i] = v; mejor[i] = b;
    }
    dibujar(dist, previo, via, mejor);
  }

  function dibujar(dist, previo, via, mejor) {
    capaIso.clearLayers();
    const centro = [origen.lat, origen.lng];
    I.umbrales.forEach((u) => {
      const estilo = { renderer: render[u], stroke: false, fill: true,
        fillColor: I.colores[u], fillOpacity: 1, interactive: false };
      L.circle(centro, { ...estilo, radius: radioCaminando(u) }).addTo(capaIso);
      for (let i = 0; i < N; i++) {
        if (llegada[i] < u) {
          L.circle([I.paradas[i][2], I.paradas[i][3]],
            { ...estilo, radius: radioCaminando(u - llegada[i]) }).addTo(capaIso);
        }
      }
    });
    // Recorre hacia atrás el camino de cada parada alcanzada y dibuja cada tramo una vez.
    const visto = new Uint8Array(dist.length);
    lineas = Object.fromEntries(I.umbrales.map((u) => [u, L.layerGroup()]));
    const rutasPorBanda = Object.fromEntries(I.umbrales.map((u) => [u, new Set()]));
    for (let i = 0; i < N; i++) {
      let estado = mejor[i];
      while (estado >= 0 && !visto[estado]) {
        visto[estado] = 1;
        const k = via[estado], llega = dist[estado], color = I.colores[banda(llega)];
        const desde = previo[estado];
        if (k >= 0) {
          const t = I.tramos[k];
          I.umbrales.forEach((u) => { if (llega <= u) rutasPorBanda[u].add(t[2]); });
          const a = I.paradas[t[0]], b = I.paradas[t[1]];
          const coords = t[4] >= 0 ? I.geom[t[4]] : [[a[2], a[3]], [b[2], b[3]]];
          L.polyline(coords, { renderer: renderLineas, color, weight: 3, opacity: 0.9,
            interactive: false }).addTo(lineas[banda(llega)]);
        } else if (k === -2 && desde >= 0) {
          const a = I.paradas[Math.floor(desde / SL)], b = I.paradas[Math.floor(estado / SL)];
          L.polyline([[a[2], a[3]], [b[2], b[3]]], { renderer: renderLineas,
            color: "#374151", weight: 2, dashArray: "3 4", interactive: false })
            .addTo(lineas[banda(llega)]);
        }
        estado = desde;
      }
    }
    const filas = I.umbrales.map((u) => {
      let paradas = 0, lejos = 0;
      for (let i = 0; i < N; i++) {
        if (i === origen.parada || llegada[i] > u) continue;
        paradas++;
        lejos = Math.max(lejos, distM(origen.lat, origen.lng, I.paradas[i][2], I.paradas[i][3]));
      }
      return "<tr data-u='" + u + "' title='Ver solo estas paradas'><td><span style='display:inline-block;width:10px;height:10px;" +
        "margin-right:4px;vertical-align:-1px;background:" + I.colores[u] + "'></span>" +
        u + " min</td><td>" + paradas.toLocaleString("es-MX") + "</td><td>" +
        rutasPorBanda[u].size + "</td><td>" + (lejos / 1000).toFixed(1) + " km</td></tr>";
    });
    cuerpoTabla.innerHTML = filas.join("");
    aplicarFiltro();
  }

  // Filtro por banda: con "acumulado" se ven las paradas a u minutos o menos
  // (igual que la tabla); sin él, solo las que llegan entre la banda anterior y u.
  const bandaVisible = (u) => {
    const sel = Number(filtro.value);
    return !sel || (acumulado.checked ? u <= sel : u === sel);
  };
  function aplicarFiltro() {
    const sel = Number(filtro.value);
    [...filtro.options].slice(1).forEach((o) => {
      const u = Number(o.value);
      o.textContent = acumulado.checked ? "≤ " + u + " min" : desdeBanda(u) + "–" + u + " min";
    });
    I.umbrales.forEach((u) => {
      const ver = bandaVisible(u);
      mapa.getPane("iso" + u).style.display = ver ? "" : "none";
      if (ver) capaIso.addLayer(lineas[u]); else capaIso.removeLayer(lineas[u]);
    });
    visibles = [];
    marcadores.forEach((m, i) => {
      const t = llegada[i];
      const ver = Number.isFinite(t) && bandaVisible(banda(t));
      m.setStyle(ver
        ? { radius: 3.5, fillColor: I.colores[banda(t)], color: "#1f2937", fillOpacity: 1 }
        : { radius: 2, fillColor: "#d1d5db", color: "#4b5563", fillOpacity: 0.9 });
      if (ver && i !== origen.parada) visibles.push(i);
    });
    visibles.sort((a, b) => llegada[a] - llegada[b]);
    cuerpoTabla.querySelectorAll("tr").forEach((tr) => {
      tr.classList.toggle("iso-sel", Number(tr.dataset.u) === sel);
    });
    tituloLista.textContent = (!sel ? "Paradas a ≤ " + T_MAX + " min"
      : acumulado.checked ? "Paradas a ≤ " + sel + " min"
      : "Paradas entre " + desdeBanda(sel) + " y " + sel + " min") +
      " (" + visibles.length.toLocaleString("es-MX") + ")";
    cuerpoLista.innerHTML = visibles.length ? visibles.map((i) => {
      const p = I.paradas[i], t = llegada[i];
      return "<div class='iso-fila' data-i='" + i + "'><span style='background:" +
        I.colores[banda(t)] + "'></span><b>" + t.toFixed(1) + "</b> min · " + esc(p[1]) +
        " <i>[" + esc(p[0]) + "]</i></div>";
    }).join("") : "<div class='iso-fila'>Ninguna parada en este rango.</div>";
  }

  function fijarOrigen(punto, nombre, parada) {
    origen = { lat: punto.lat, lng: punto.lng, nombre, parada };
    etiquetaOrigen.textContent = nombre;
    if (marcadorOrigen) mapa.removeLayer(marcadorOrigen);
    marcadorOrigen = L.circleMarker(punto, { radius: 7, color: "#111827", weight: 3,
      fillColor: "#ffffff", fillOpacity: 1, pane: "isoParadas", interactive: false }).addTo(mapa);
    calcular();
  }
  function elegirParada(i) {
    const p = I.paradas[i];
    fijarOrigen(L.latLng(p[2], p[3]), p[1] + " (parada " + p[0] + ")", i);
    mapa.panTo([p[2], p[3]]);
  }
  function elegirEstacion() {
    fijarOrigen(estacion, "Estación Corregidora", null);
    mapa.panTo(estacion);
  }

  document.getElementById("iso-estacion").addEventListener("click", elegirEstacion);
  buscador.addEventListener("change", () => {
    const i = porOpcion.get(buscador.value);
    if (i != null) { elegirParada(i); buscador.value = ""; }
  });
  [escenario, penalizacion, usarPie].forEach((c) => c.addEventListener("change", calcular));
  [filtro, acumulado].forEach((c) => c.addEventListener("change", aplicarFiltro));
  cuerpoTabla.addEventListener("click", (e) => {
    const fila = e.target.closest("tr[data-u]");
    if (!fila) return;
    filtro.value = filtro.value === fila.dataset.u ? "" : fila.dataset.u;
    aplicarFiltro();
  });
  cuerpoLista.addEventListener("click", (e) => {
    const fila = e.target.closest("[data-i]");
    if (!fila) return;
    const i = Number(fila.dataset.i), p = I.paradas[i];
    mapa.setView([p[2], p[3]], Math.max(mapa.getZoom(), 15));
    marcadores[i].openTooltip();
  });
  // Botón "Isócrona desde esta parada" de los popups de transbordo.
  mapa.on("popupopen", (e) => {
    const boton = e.popup.getElement().querySelector("[data-iso-origen]");
    if (!boton) return;
    boton.addEventListener("click", () => {
      const i = indice[boton.dataset.isoOrigen];
      if (i != null) { mapa.closePopup(); elegirParada(i); }
    });
  });

  mapa.setView(estacion, 12);
  elegirEstacion();
})();
"""


def datos_isocronas_js(stop_times, trips, stops, routes, calibracion, trazos_viales):
    """Prepara el grafo compacto que usa JS_ISOCRONAS.

    - Tramos: tiempo programado mediano entre paradas consecutivas por ruta,
      con todos los viajes del GTFS (el mismo grafo del Dijkstra de 01 y 02).
    - Rutas: constante y factor P50/P80 de la calibración con Google; las rutas
      sin modelo propio usan el global.
    - Geometría: el trazo por calles de cada tramo, si existe.
    - Caminata: pares de paradas a RADIO_CONEXION_A_PIE_M o menos en línea recta.
    """
    segmentos = segmentos_medianos_gtfs(stop_times, trips)
    segmentos["route_id"] = segmentos["route_id"].astype(str)
    ids = sorted(set(segmentos["stop_id"].astype(str)) | set(segmentos["next_stop_id"].astype(str)))
    datos_parada = stops.assign(stop_id=stops["stop_id"].astype(str)).set_index("stop_id")
    ids = [s for s in ids if s in datos_parada.index]
    indice = {s: i for i, s in enumerate(ids)}
    paradas = [
        [s, str(datos_parada.at[s, "stop_name"]),
         round(float(datos_parada.at[s, "stop_lat"]), 6),
         round(float(datos_parada.at[s, "stop_lon"]), 6)]
        for s in ids
    ]

    calibracion = calibracion.assign(route_id=calibracion["route_id"].astype(str))
    modelo_global = calibracion.loc[calibracion["route_id"].eq("__GLOBAL__")].iloc[0]
    modelos = calibracion.set_index("route_id")
    nombres = routes.assign(route_id=routes["route_id"].astype(str)).set_index("route_id")["route_short_name"]
    route_ids = sorted(segmentos["route_id"].unique())
    indice_ruta = {r: i for i, r in enumerate(route_ids)}
    rutas = []
    for r in route_ids:
        modelo = modelos.loc[r] if r in modelos.index else modelo_global
        rutas.append([
            r, str(nombres.get(r, r)),
            round(float(modelo["constante_p50_min"]), 3), round(float(modelo["factor_p50"]), 4),
            round(float(modelo["constante_p80_min"]), 3), round(float(modelo["factor_p80"]), 4),
        ])

    geom, indice_geom, tramos = [], {}, []
    for fila in segmentos.itertuples(index=False):
        a, b = str(fila.stop_id), str(fila.next_stop_id)
        if a not in indice or b not in indice:
            continue
        if (a, b) not in indice_geom:
            trazo = trazos_viales.get((a, b))
            indice_geom[(a, b)] = len(geom) if trazo else -1
            if trazo:
                geom.append([[round(la, 5), round(lo, 5)] for la, lo in trazo["coordenadas"]])
        tramos.append([
            indice[a], indice[b], indice_ruta[fila.route_id],
            round(float(fila.tiempo_segmento_min), 2), indice_geom[(a, b)],
        ])

    lat = np.array([p[2] for p in paradas])
    lon = np.array([p[3] for p in paradas])
    pie = []
    for inicio in range(0, len(paradas), 400):
        fin = min(inicio + 400, len(paradas))
        distancias = haversine_m(lat[inicio:fin, None], lon[inicio:fin, None], lat[None, :], lon[None, :])
        filas, columnas = np.nonzero(distancias <= RADIO_CONEXION_A_PIE_M)
        for fila, columna in zip(filas, columnas):
            if inicio + fila < columna:
                pie.append([int(inicio + fila), int(columna), round(float(distancias[fila, columna]))])

    return {
        "paradas": paradas,
        "rutas": rutas,
        "tramos": tramos,
        "geom": geom,
        "pie": pie,
        "umbrales": UMBRALES_ISOCRONA_MIN,
        "colores": {str(u): c for u, c in COLORES_ISOCRONA.items()},
        "velocidad": VELOCIDAD_CAMINATA_M_MIN,
        "desvio": FACTOR_DESVIO_CAMINATA,
        "radio": RADIO_BUSQUEDA_PARADA_M,
        "estacion": [CORREGIDORA_LAT, CORREGIDORA_LON],
    }


PANEL_ISOCRONAS = """
<div id="panel-isocronas" style="position:fixed;top:10px;left:54px;z-index:10000;width:265px;max-width:calc(100vw - 70px);max-height:calc(100vh - 20px);overflow-y:auto;background:#fff;border:1px solid #9ca3af;border-radius:8px;padding:8px 9px;box-shadow:0 2px 10px #0003;font:11px/1.3 Arial,sans-serif;color:#111827">
  <style>
    #panel-isocronas table{width:100%;border-collapse:collapse;margin-top:6px}
    #panel-isocronas th{background:#17324d;color:#fff;font-weight:600;padding:3px 4px;text-align:right}
    #panel-isocronas th:first-child,#panel-isocronas td:first-child{text-align:left}
    #panel-isocronas td{padding:3px 4px;border-bottom:1px solid #e5ebf0;text-align:right;white-space:nowrap}
    #panel-isocronas label{display:block;margin-top:5px}
    #panel-isocronas input,#panel-isocronas select{font:inherit}
    #iso-tabla tr{cursor:pointer}
    #iso-tabla tr:hover{background:#eef6f3}
    #iso-tabla tr.iso-sel{background:#dbeafe;font-weight:700}
    #iso-lista{max-height:170px;overflow-y:auto;border:1px solid #e5ebf0;margin-top:3px}
    #iso-lista .iso-fila{padding:2px 4px;border-bottom:1px solid #f1f5f9;cursor:pointer}
    #iso-lista .iso-fila:hover{background:#eef6f3}
    #iso-lista .iso-fila span{display:inline-block;width:8px;height:8px;margin-right:4px;border:1px solid #1f2937;border-radius:50%}
    #iso-lista .iso-fila i{color:#6b7280;font-style:normal}
  </style>
  <strong style="font-size:13px">Isócronas: ¿hasta dónde llego?</strong>
  <div style="margin:5px 0 3px">Origen: <b id="iso-origen">Estación Corregidora</b></div>
  <button id="iso-estacion" type="button" style="padding:3px 7px;cursor:pointer">Usar Estación Corregidora</button>
  <input id="iso-buscar" list="iso-paradas" placeholder="Buscar parada (nombre o ID)" style="width:100%;box-sizing:border-box;margin-top:5px;padding:3px">
  <datalist id="iso-paradas"></datalist>
  <label>Tiempos
    <select id="iso-escenario" style="margin-left:4px">
      <option value="p80" selected>P80 confiable</option>
      <option value="p50">P50 habitual</option>
      <option value="gtfs">Horario GTFS</option>
    </select>
  </label>
  <label>Transbordo (min)
    <input id="iso-penalizacion" type="number" min="0" step="1" value="__PENALIZACION__" style="width:45px;margin-left:4px">
  </label>
  <label><input id="iso-pie" type="checkbox"> Transbordar caminando entre paradas ≤ __RADIO_PIE__ m</label>
  <table>
    <thead><tr><th>Tiempo</th><th>Paradas</th><th>Rutas</th><th>Hasta</th></tr></thead>
    <tbody id="iso-tabla"></tbody>
  </table>
  <label>Ver paradas
    <select id="iso-filtro" style="margin-left:4px"><option value="">Todas</option></select>
  </label>
  <label><input id="iso-acumulado" type="checkbox" checked> Incluir las de menos tiempo (como la tabla)</label>
  <div style="margin-top:6px"><b id="iso-lista-titulo"></b></div>
  <div id="iso-lista"></div>
  <div style="margin-top:6px;color:#6b7280">
    Clic en cualquier parada del mapa para usarla como origen; clic en una fila de la
    tabla para ver solo esas paradas y en una parada de la lista para ubicarla. Paradas y rutas alcanzables en ese
    tiempo o menos; "Hasta" es la parada más lejana en línea recta. La mancha suma lo que
    se alcanza caminando (__VELOCIDAD__ km/h) desde cada parada con el tiempo que sobra.
  </div>
</div>
"""


def datos_conexiones_js(servicio, conexiones, paradas_por_trip, trip_ids,
                        nombres_ruta, colores_ruta):
    """Prepara el bloque de datos compacto que usa JS_CONEXIONES."""
    servicio = servicio.assign(
        route_id=servicio["route_id"].astype(str),
        sentido=servicio["direction_key"].astype(str).str.replace("sin_sentido", "?"),
    )
    paradas = {}
    for stop_id, grupo in servicio.groupby("stop_id", sort=False):
        primera = grupo.iloc[0]
        paradas[str(stop_id)] = [
            str(primera["stop_name"]),
            round(float(primera["stop_lat"]), 6),
            round(float(primera["stop_lon"]), 6),
            grupo[["route_id", "sentido"]].values.tolist(),
        ]
    a_pie = conexiones.loc[conexiones["tipo_conexion"].eq("A pie")]
    apie = {
        str(stop_id): [
            [str(fila.stop_id_conexion), round(fila.distancia_m), list(fila.route_ids_conexion)]
            for fila in grupo.itertuples(index=False)
        ]
        for stop_id, grupo in a_pie.groupby("stop_id", sort=False)
    }
    # Cada par se dibuja una sola vez aunque conecte en ambos sentidos.
    pares = {}
    for fila in a_pie.itertuples(index=False):
        clave = tuple(sorted((str(fila.stop_id), str(fila.stop_id_conexion))))
        pares.setdefault(clave, [*clave, round(fila.distancia_m)])
    misma = conexiones.loc[
        conexiones["tipo_conexion"].eq("Misma parada"), "stop_id"
    ].astype(str).tolist()
    return {
        "rutas": {r: [nombres_ruta[r], colores_ruta[r]] for r in nombres_ruta},
        "paradas": paradas,
        "apie": apie,
        "pares": list(pares.values()),
        "misma": misma,
        "solo_pie": sorted(set(apie) - set(misma)),
        "trips": {t: paradas_por_trip.get(t, []) for t in trip_ids},
        "radio": RADIO_CONEXION_A_PIE_M,
        "velocidad": VELOCIDAD_CAMINATA_M_MIN,
        "desvio": FACTOR_DESVIO_CAMINATA,
    }


def crear_mapa_tiempos_totales(trazos, salida_html, servicio=None,
                               conexiones=None, paradas_por_trip=None,
                               isocronas=None):
    """Dibuja una línea por ruta-sentido con GTFS, P50/P80 y sus conexiones.

    Con `servicio`, `conexiones` y `paradas_por_trip` (de calcular_conexiones)
    agrega dos capas: paradas donde se cruzan rutas y conexiones a pie. Al dar
    clic en una ruta se listan sus paradas de transbordo.

    Con `isocronas` (de datos_isocronas_js) el mapa abre en modo isócronas desde
    la Estación Corregidora; las rutas y conexiones quedan como capas opcionales.
    """
    if not trazos:
        raise ValueError("No existen trazos calculables para crear el mapa.")
    todas_coords = [coord for trazo in trazos for coord in trazo["coordenadas"]]
    centro = [
        float(np.mean([coord[0] for coord in todas_coords])),
        float(np.mean([coord[1] for coord in todas_coords])),
    ]
    mapa = folium.Map(location=centro, zoom_start=11, tiles="OpenStreetMap")
    capa = folium.FeatureGroup(
        name=f"Rutas con tiempo GTFS y calibrado ({len(trazos)})",
        show=isocronas is None,
    )
    colores_ruta = colores_por_ruta(trazo["route_id"] for trazo in trazos)
    nombres_ruta = {str(t["route_id"]): str(t["route_short_name"]) for t in trazos}
    hay_conexiones = servicio is not None and conexiones is not None

    elementos_leyenda = []
    lineas_js = []
    for trazo in sorted(
        trazos, key=lambda x: (x["route_short_name"], x["direction_key"])
    ):
        color = colores_ruta[str(trazo["route_id"])]
        sentido = trazo["direction_key"].replace("sin_sentido", "sin dato")
        etiqueta = (
            f"Ruta {trazo['route_short_name']} · sentido {sentido} | "
            f"{trazo['distancia_km']:.1f} km por calles | "
            f"GTFS P50 {trazo['p50_min']:.1f} min | "
            f"estimado P50 {trazo['estimado_p50_min']:.1f} min | "
            f"estimado P80 {trazo['estimado_p80_min']:.1f} min | "
            f"Google n={trazo['observaciones_google']} ({trazo['fuente_modelo']})"
        )
        if hay_conexiones:
            etiqueta += " | clic: ver conexiones"
        linea = folium.PolyLine(
            trazo["coordenadas"], color=color, weight=4, opacity=0.78,
            tooltip=etiqueta,
        )
        linea.add_to(capa)
        encabezado = (
            f"<b>Ruta {html.escape(trazo['route_short_name'])} · sentido "
            f"{html.escape(sentido)}</b><br>"
            f"GTFS P50 {trazo['p50_min']:.1f} min · estimado P50 "
            f"{trazo['estimado_p50_min']:.1f} · P80 {trazo['estimado_p80_min']:.1f} min"
        )
        lineas_js.append(
            (linea.get_name(), trazo["trip_id"], str(trazo["route_id"]), encabezado)
        )
        elementos_leyenda.append(
            "<tr>"
            f"<td style='padding:5px 8px;white-space:nowrap'>"
            f"<span style='display:inline-block;width:12px;height:4px;"
            f"background:{color};margin-right:6px;vertical-align:middle'></span>"
            f"{html.escape(str(trazo['route_short_name']))}</td>"
            f"<td style='padding:5px 8px;white-space:nowrap'>S{html.escape(sentido)}</td>"
            f"<td style='padding:5px 8px;text-align:right;white-space:nowrap'>{trazo['p50_min']:.1f} min</td>"
            f"<td style='padding:5px 8px;text-align:right;white-space:nowrap'>{trazo['estimado_p50_min']:.1f} min</td>"
            f"<td style='padding:5px 8px;text-align:right;white-space:nowrap'>{trazo['estimado_p80_min']:.1f} min</td>"
            "</tr>"
        )
    capa.add_to(mapa)

    if hay_conexiones:
        datos = datos_conexiones_js(
            servicio, conexiones, paradas_por_trip or {},
            [t["trip_id"] for t in trazos], nombres_ruta, colores_ruta,
        )
        capa_misma = folium.FeatureGroup(
            name=f"Paradas de transbordo: misma parada ({len(datos['misma'])})",
            show=isocronas is None,
        ).add_to(mapa)
        capa_pie = folium.FeatureGroup(
            name=f"Conexiones a pie ≤ {RADIO_CONEXION_A_PIE_M:g} m ({len(datos['pares'])} pares)",
            show=False,
        ).add_to(mapa)

    folium.LayerControl(collapsed=False).add_to(mapa)
    if hay_conexiones:
        lineas_literal = "[" + ",".join(
            f"[{nombre},{json_para_script(trip)},{json_para_script(ruta)},"
            f"{json_para_script(encabezado)}]"
            for nombre, trip, ruta, encabezado in lineas_js
        ) + "]"
        codigo = (
            JS_CONEXIONES.replace("__DATOS__", json_para_script(datos))
            .replace("__CAPA_MISMA__", capa_misma.get_name())
            .replace("__CAPA_PIE__", capa_pie.get_name())
            .replace("__LINEAS__", lineas_literal)
            .replace("__MAPA__", mapa.get_name())
        )
        mapa.add_child(ScriptMapa(codigo))
    if isocronas is not None:
        mapa.add_child(ScriptMapa(
            JS_ISOCRONAS.replace("__ISO__", json_para_script(isocronas))
            .replace("__MAPA__", mapa.get_name())
        ))
        mapa.get_root().html.add_child(folium.Element(
            PANEL_ISOCRONAS.replace("__PENALIZACION__", f"{PENALIZACION_TRANSBORDO_MIN:g}")
            .replace("__RADIO_PIE__", f"{RADIO_CONEXION_A_PIE_M:g}")
            .replace("__VELOCIDAD__", f"{VELOCIDAD_CAMINATA_M_MIN * 60 / 1000:g}")
        ))
    else:
        mapa.fit_bounds(todas_coords)
    nota_conexiones = (
        "<div style='margin:4px 0;color:#374151'>"
        "<span style='display:inline-block;width:9px;height:9px;border-radius:50%;"
        "background:#fff;border:2px solid #111827;margin-right:5px;vertical-align:middle'></span>"
        "Transbordo en la misma parada · clic en una ruta o parada para ver "
        "a dónde conecta</div>"
        if hay_conexiones else ""
    )
    leyenda = folium.Element(
        "<details class='leyenda-tiempos' style='position:fixed;bottom:24px;z-index:9999;"
        + ("right:12px;width:min(590px,calc(100vw - 345px));" if isocronas is not None
           else "left:24px;width:min(590px,calc(100vw - 32px));")
        + "max-height:38vh;overflow:auto;"
        "background:#fff;border:1px solid #d1dbe5;border-radius:9px;padding:10px 12px;"
        "box-shadow:0 3px 12px #0f172a33;font:12px/1.35 Arial,sans-serif;color:#1f2937'>"
        "<style>.leyenda-tiempos table{width:100%;border-collapse:collapse;font-size:11px}"
        ".leyenda-tiempos thead{position:sticky;top:0;z-index:2;background:#17324d;color:#fff}"
        ".leyenda-tiempos th{text-align:right;padding:7px 8px;border-bottom:2px solid #0f766e;white-space:nowrap}"
        ".leyenda-tiempos th:first-child,.leyenda-tiempos th:nth-child(2){text-align:left}"
        ".leyenda-tiempos tbody tr{border-bottom:1px solid #e5ebf0}"
        ".leyenda-tiempos tbody tr:nth-child(even){background:#f3f7fa}"
        ".leyenda-tiempos tbody tr:hover{background:#e9f6f2}</style>"
        "<summary style='font-weight:700;font-size:13px;color:#17324d;cursor:pointer'>"
        "Tiempos GTFS y calibrados por ruta y sentido</summary>"
        + nota_conexiones
        + "<div style='overflow-x:auto'><table><thead><tr>"
        + "<th>Ruta</th><th>Sentido</th><th>GTFS</th><th>P50</th><th>P80</th>"
        + "</tr></thead><tbody>"
        + "".join(elementos_leyenda)
        + "</tbody></table></div></details>"
    )
    mapa.get_root().html.add_child(leyenda)
    salida_html.parent.mkdir(parents=True, exist_ok=True)
    mapa.save(salida_html)
    return mapa


## 1. Carga del GTFS


In [8]:
stop_times = pd.read_csv(
    GTFS_DIR / "stop_times.txt",
    dtype={"trip_id": "string", "stop_id": "string"},
)
trips = pd.read_csv(
    GTFS_DIR / "trips.txt",
    dtype={"trip_id": "string", "route_id": "string"},
)
routes = pd.read_csv(GTFS_DIR / "routes.txt", dtype={"route_id": "string"})
stops = pd.read_csv(GTFS_DIR / "stops.txt", dtype={"stop_id": "string"})
print(f"Stop times: {len(stop_times):,}")
print(f"Viajes catalogados: {trips['trip_id'].nunique():,}")
print(f"Rutas catalogadas: {routes['route_id'].nunique():,}")


Stop times: 1,326,009
Viajes catalogados: 32,536
Rutas catalogadas: 217


## 2. Duración completa de cada viaje

Para cada `trip_id` se resta la salida de la primera parada a la llegada de la última. Las horas GTFS mayores a 24:00 se conservan correctamente.


In [9]:
viajes = calcular_duraciones_viajes(stop_times, trips, routes)
resumen = agregar_rutas_sin_viajes(
    resumir_por_ruta_sentido(viajes), routes
)
resumen.to_csv(RESUMEN_PATH, index=False)
viajes.to_csv(VIAJES_PATH, index=False)
print(f"Viajes analizados: {viajes['trip_id'].nunique():,}")
print(f"Rutas con viajes: {viajes['route_id'].nunique():,}")
print(
    "Rutas del catálogo sin viajes: "
    f"{resumen.loc[resumen['estado_calculo'].ne('calculado'), 'route_id'].nunique():,}"
)
display(estilizar_tabla(
    resumen.loc[resumen["estado_calculo"].eq("calculado")].head(10),
    "Duraciones GTFS por ruta y sentido",
))


Viajes analizados: 32,536
Rutas con viajes: 117
Rutas del catálogo sin viajes: 100


route_id,route_short_name,route_long_name,direction_id,numero_viajes,numero_servicios,duracion_min_min,duracion_p10_min,duracion_mediana_min,duracion_promedio_min,duracion_p90_min,duracion_max_min,paradas_mediana,estado_calculo
386,AIQ01,Alameda - Aeropuerto Internacional De Queretaro,0.00,33,3,55.30,55.30,73.73,67.70,73.73,73.73,21.00,calculado
386,AIQ01,Alameda - Aeropuerto Internacional De Queretaro,1.00,33,3,52.63,52.63,70.02,63.70,70.02,70.02,28.00,calculado
96,C21,Las Flores - Fracc Fray Junipero Serra,0.00,204,3,93.22,93.22,104.88,102.89,109.30,109.30,69.00,calculado
96,C21,Las Flores - Fracc Fray Junipero Serra,1.00,204,3,93.73,93.73,98.28,98.95,103.87,103.87,64.00,calculado
97,C22,Fracc Santuarios - Penal,0.00,160,3,76.47,79.98,90.67,89.11,95.12,95.12,67.00,calculado
97,C22,Fracc Santuarios - Penal,1.00,160,3,79.73,85.18,94.78,93.83,101.60,101.60,70.00,calculado
98,C23,Lomas De La Cruz - Terminal De Autobuses (taq),0.00,188,3,91.78,91.78,98.15,96.74,99.87,99.87,69.00,calculado
98,C23,Lomas De La Cruz - Terminal De Autobuses (taq),1.00,188,3,80.47,80.47,90.95,88.05,91.85,91.85,65.00,calculado
319,C24,La Pradera - Bosques Del Cimatario,0.00,173,3,65.08,65.08,69.65,69.93,74.20,74.20,58.00,calculado
319,C24,La Pradera - Bosques Del Cimatario,1.00,175,3,60.77,60.77,73.45,69.54,73.45,73.45,51.00,calculado


## 3. Aprendizaje de la constante y el factor de atraso

Cada observación histórica compara una ETA de Google con el tiempo GTFS hacia Corregidora. Se ajustan dos modelos: `P50 = constante_P50 + factor_P50 × GTFS` y `P80 = constante_P80 + factor_P80 × GTFS`. La constante se aplica una sola vez por viaje y el factor se aplica a cada segmento.

La atribución por ruta usa `route_id_principal` del camino GTFS; Google no confirma en la matriz que haya utilizado esa misma línea. Por ello la calibración es un **proxy**. Las rutas con muestra insuficiente utilizan el modelo global y las demás se suavizan hacia él para reducir sobreajuste.


In [10]:
if OBSERVACIONES_GOOGLE_PATH.exists():
    observaciones_crudas = pd.read_csv(
        OBSERVACIONES_GOOGLE_PATH, dtype={"stop_id": "string"}
    )
    observaciones_modelo = preparar_observaciones_google(observaciones_crudas)
    calibracion = calibrar_modelos_por_ruta(observaciones_modelo)
else:
    observaciones_modelo = pd.DataFrame()
    calibracion = pd.DataFrame([{
        "route_id": "__GLOBAL__", "observaciones_google": 0,
        "rango_gtfs_observado_min": 0.0, "peso_modelo_ruta": 0.0,
        "constante_p50_min": 0.0, "factor_p50": 1.0,
        "constante_p80_min": 0.0, "factor_p80": 1.0,
        "fuente_modelo": "GTFS sin histórico de Google",
    }])

modelo_global = calibracion.loc[calibracion["route_id"].eq("__GLOBAL__")].iloc[0]
modelos_ruta = calibracion.loc[calibracion["route_id"].ne("__GLOBAL__")].copy()
resumen["route_id"] = resumen["route_id"].astype("string")
resumen = resumen.merge(modelos_ruta, on="route_id", how="left", validate="many_to_one")
for columna in ["constante_p50_min", "factor_p50", "constante_p80_min", "factor_p80"]:
    resumen[columna] = resumen[columna].fillna(float(modelo_global[columna]))
resumen["observaciones_google"] = resumen["observaciones_google"].fillna(0).astype(int)
resumen["rango_gtfs_observado_min"] = resumen["rango_gtfs_observado_min"].fillna(0.0)
resumen["peso_modelo_ruta"] = resumen["peso_modelo_ruta"].fillna(0.0)
resumen["fuente_modelo"] = resumen["fuente_modelo"].fillna("global")
resumen["tiempo_operativo_p50_min"] = (
    resumen["duracion_mediana_min"] * resumen["factor_p50"]
)
resumen["tiempo_operativo_p80_min"] = (
    resumen["duracion_mediana_min"] * resumen["factor_p80"]
)
resumen["tiempo_pasajero_p50_min"] = (
    resumen["constante_p50_min"] + resumen["tiempo_operativo_p50_min"]
)
resumen["tiempo_pasajero_p80_min"] = (
    resumen["constante_p80_min"] + resumen["tiempo_operativo_p80_min"]
)
columnas_estimadas = [
    "constante_p50_min", "factor_p50", "constante_p80_min", "factor_p80",
    "tiempo_operativo_p50_min", "tiempo_operativo_p80_min",
    "tiempo_pasajero_p50_min", "tiempo_pasajero_p80_min",
]
resumen[columnas_estimadas] = resumen[columnas_estimadas].round(2)
calibracion.to_csv(CALIBRACION_PATH, index=False)
resumen.to_csv(RESUMEN_PATH, index=False)
print(f"Observaciones válidas usadas: {len(observaciones_modelo):,}")
print(f"Rutas con calibración propia suavizada: {len(modelos_ruta):,}")
print(
    f"Modelo global P50: {modelo_global['constante_p50_min']:.2f} + "
    f"{modelo_global['factor_p50']:.3f} × GTFS"
)
print(
    f"Modelo global P80: {modelo_global['constante_p80_min']:.2f} + "
    f"{modelo_global['factor_p80']:.3f} × GTFS"
)
display(estilizar_tabla(
    resumen.loc[resumen["estado_calculo"].eq("calculado"), [
        "route_short_name", "direction_id", "duracion_mediana_min",
        "tiempo_pasajero_p50_min", "tiempo_pasajero_p80_min",
        "observaciones_google", "fuente_modelo",
    ]].head(10),
    "Tiempos estimados y calibración por ruta",
))


Observaciones válidas usadas: 9,216
Rutas con calibración propia suavizada: 54
Modelo global P50: 0.00 + 1.000 × GTFS
Modelo global P80: 0.00 + 1.000 × GTFS


route_short_name,direction_id,duracion_mediana_min,tiempo_pasajero_p50_min,tiempo_pasajero_p80_min,observaciones_google,fuente_modelo
AIQ01,0.00,73.73,73.73,73.73,0,global
AIQ01,1.00,70.02,70.02,70.02,0,global
C21,0.00,104.88,104.88,104.88,124,ruta suavizada con global
C21,1.00,98.28,98.28,98.28,124,ruta suavizada con global
C22,0.00,90.67,90.67,90.67,168,ruta suavizada con global
C22,1.00,94.78,94.78,94.78,168,ruta suavizada con global
C23,0.00,98.15,98.15,98.15,106,ruta suavizada con global
C23,1.00,90.95,90.95,90.95,106,ruta suavizada con global
C24,0.00,69.65,69.65,69.65,188,ruta suavizada con global
C24,1.00,73.45,73.45,73.45,188,ruta suavizada con global


## 4. Propagación por segmentos hacia la primera parada

Para cada ruta y sentido se elige el viaje más cercano a la mediana GTFS. Cada segmento se multiplica por su factor P50/P80 y luego se realiza una suma acumulada inversa desde la última parada. La constante se añade una sola vez al tiempo del pasajero; no se repite en cada segmento. Los segmentos corregidos son estimaciones modeladas: la matriz actual de Google entrega tiempos completos origen-destino, no observaciones independientes de cada segmento.


In [11]:
representativos = seleccionar_viajes_representativos(viajes, resumen)
tiempos_acumulados = construir_tiempos_acumulados(
    stop_times, stops, representativos
)
tiempos_acumulados.to_csv(ACUMULADOS_PATH, index=False)
assert tiempos_acumulados["tiempo_restante_operativo_p50_min"].ge(
    tiempos_acumulados["tiempo_restante_gtfs_min"] - 1e-9
).all()
assert tiempos_acumulados["tiempo_restante_pasajero_p80_min"].ge(
    tiempos_acumulados["tiempo_restante_pasajero_p50_min"] - 1e-9
).all()
print(f"Paradas acumuladas exportadas: {len(tiempos_acumulados):,}")
print(f"Archivo: {ACUMULADOS_PATH}")
display(estilizar_tabla(tiempos_acumulados.head(10), "Tiempos acumulados por parada"))


Paradas acumuladas exportadas: 8,071
Archivo: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/tiempos_acumulados_paradas_rutas.csv


route_id,route_short_name_resumen,direction_key,trip_id,stop_sequence,stop_id,stop_name,next_stop_id,tiempo_segmento_gtfs_min,tiempo_segmento_p50_min,tiempo_segmento_p80_min,tiempo_restante_gtfs_min,tiempo_restante_operativo_p50_min,tiempo_restante_operativo_p80_min,tiempo_restante_pasajero_p50_min,tiempo_restante_pasajero_p80_min,constante_p50_min,factor_p50,constante_p80_min,factor_p80,observaciones_google,fuente_modelo
386,AIQ01,1,10,1,2033,Prol. Luis Pasteur/Constituyentes,3573,0.90,0.90,0.90,65.68,65.68,65.68,65.68,65.68,0.00,1.00,0.00,1.00,0,global
386,AIQ01,1,10,2,3573,Av. Luis Pateur Sur/José A. Bustamante,2034,0.37,0.37,0.37,64.78,64.78,64.78,64.78,64.78,0.00,1.00,0.00,1.00,0,global
386,AIQ01,1,10,3,2034,Prol. Luis Pasteu /Chichén Itzá,2035,0.45,0.45,0.45,64.42,64.42,64.42,64.42,64.42,0.00,1.00,0.00,1.00,0,global
386,AIQ01,1,10,4,2035,Prol. Luis Pasteur/José Sotelo,3341,1.42,1.42,1.42,63.97,63.97,63.97,63.97,63.97,0.00,1.00,0.00,1.00,0,global
386,AIQ01,1,10,5,3341,Carr. Lateral No. 57/Prol. del Cimatario,2751,0.77,0.77,0.77,62.55,62.55,62.55,62.55,62.55,0.00,1.00,0.00,1.00,0,global
386,AIQ01,1,10,6,2751,Lateral Querétaro A Mexico/Comercial Mexicana,3783,7.67,7.67,7.67,61.78,61.78,61.78,61.78,61.78,0.00,1.00,0.00,1.00,0,global
386,AIQ01,1,10,7,3783,San Isidro Miranda/Autopista México/Querétaro,3863,2.12,2.12,2.12,54.12,54.12,54.12,54.12,54.12,0.00,1.00,0.00,1.00,0,global
386,AIQ01,1,10,8,3863,Carr. Federal No. 57/Calle San Ignacio,5213,1.02,1.02,1.02,52.00,52.00,52.00,52.00,52.00,0.00,1.00,0.00,1.00,0,global
386,AIQ01,1,10,9,5213,Av. Estado de México/Av. Aguascalientes,3784,0.55,0.55,0.55,50.98,50.98,50.98,50.98,50.98,0.00,1.00,0.00,1.00,0,global
386,AIQ01,1,10,10,3784,Carr. Federal No. 57/Calle Jalisco,3864,1.10,1.10,1.10,50.43,50.43,50.43,50.43,50.43,0.00,1.00,0.00,1.00,0,global


## 5. Mapa de isócronas, rutas y conexiones

El mapa abre con **isócronas desde la Estación Corregidora**: muestra hasta dónde se llega en 5, 15, 30, 45 y 60 minutos combinando caminata y autobús, y el panel cuenta cuántas paradas y rutas quedan al alcance en cada tiempo y qué tan lejos está la parada más lejana.

- **Cambiar el origen:** clic en cualquier parada, búsqueda por nombre o ID, o el botón *Isócrona desde esta parada* de los popups de transbordo. El botón *Usar Estación Corregidora* regresa al origen inicial.
- **Cálculo:** desde el origen se camina a las paradas a `RADIO_BUSQUEDA_PARADA_M` o menos (línea recta × `FACTOR_DESVIO_CAMINATA`, a `VELOCIDAD_CAMINATA_KMH`). Después se recorren los tramos del GTFS con su tiempo programado mediano, igual que el Dijkstra de `01` y `02`. Al abordar la primera ruta se suma la constante de espera del escenario; cada cambio de ruta suma la penalización de transbordo.
- **Escenarios:** *P80 confiable* (predeterminado, igual que los escenarios de demanda), *P50 habitual* u *Horario GTFS* sin calibrar. P50 y P80 usan la constante y el factor de cada ruta de la sección 3, o los del modelo global si la ruta no tiene modelo propio.
- **La mancha de color** es lo que se alcanza caminando, con el tiempo que sobra, desde cada parada a la que se llegó; las líneas son los tramos de autobús usados, coloreados según la banda de tiempo.
- **Transbordos caminando** entre paradas a `RADIO_CONEXION_A_PIE_M` o menos: apagados al inicio para coincidir con el resto del análisis; se pueden activar en el panel.

Las capas anteriores siguen disponibles en el control de capas, apagadas al inicio:

Cada ruta utiliza un color diferente. El tooltip y la leyenda comparan la mediana GTFS con los tiempos estimados P50 y P80 del pasajero e indican si el modelo procede de la ruta o del ajuste global.

Además:

- **Clic en una ruta:** lista sus paradas donde se puede transbordar y a qué rutas, en la misma parada o caminando.
- **Capa *Paradas de transbordo: misma parada*:** puntos negros donde se cruzan dos o más rutas; el tamaño crece con el número de rutas. El clic muestra las rutas que pasan y las paradas cercanas a las que se puede caminar.
- **Capa *Conexiones a pie* (apagada al inicio):** líneas punteadas entre paradas cercanas que ofrecen rutas distintas.

La tabla completa se exporta a `data/processed/conexiones_paradas_rutas.csv`.

In [12]:
representativos = seleccionar_viajes_representativos(viajes, resumen)
if TRAZOS_PATH.exists():
    trazos_viales = cargar_trazos(TRAZOS_PATH)
else:
    trazos_viales = {}
    print(
        "Falta segmentos_red_vial.csv: las rutas se dibujan en línea recta. "
        "Ejecute 01_PromedioRutas.ipynb para trazarlas sobre las calles."
    )
trazos = construir_trazos_representativos(
    stop_times, stops, representativos, trazos_viales
)
servicio_paradas, conexiones, paradas_por_trip = calcular_conexiones(
    stop_times, stops, representativos, RADIO_CONEXION_A_PIE_M
)
conexiones.assign(
    route_ids_conexion=conexiones["route_ids_conexion"].map(" ".join)
).to_csv(CONEXIONES_PATH, index=False, encoding="utf-8-sig")
misma_parada = conexiones[conexiones["tipo_conexion"].eq("Misma parada")]
a_pie = conexiones[conexiones["tipo_conexion"].eq("A pie")]
print(f"Paradas donde se cruzan 2 o más rutas: {len(misma_parada):,}")
print(f"Paradas con al menos una conexión a pie: {a_pie['stop_id'].nunique():,}")
print(f"Conexiones exportadas: {CONEXIONES_PATH}")
display(estilizar_tabla(
    misma_parada.assign(numero_rutas=misma_parada["route_ids_conexion"].map(len))
    .sort_values(["numero_rutas", "stop_name"], ascending=[False, True])
    [["stop_id", "stop_name", "numero_rutas", "rutas_en_parada"]]
    .head(10),
    "Paradas con más rutas conectadas",
))

datos_isocronas = datos_isocronas_js(
    stop_times, trips, stops, routes, calibracion, trazos_viales
)
print(
    f"Isócronas: {len(datos_isocronas['paradas']):,} paradas, "
    f"{len(datos_isocronas['tramos']):,} tramos de ruta y "
    f"{len(datos_isocronas['pie']):,} pares de paradas caminables."
)
mapa_tiempos = crear_mapa_tiempos_totales(
    trazos, MAPA_PATH, servicio_paradas, conexiones, paradas_por_trip,
    isocronas=datos_isocronas,
)
print(f"Combinaciones ruta-sentido dibujadas: {len(trazos):,}")
print(f"Copia HTML: {MAPA_PATH}")
mapa_tiempos


Paradas donde se cruzan 2 o más rutas: 1,527
Paradas con al menos una conexión a pie: 1,148
Conexiones exportadas: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/conexiones_paradas_rutas.csv


stop_id,stop_name,numero_rutas,rutas_en_parada
1921,Avenida Ignacio Zaragoza/Comercial Mexicana,32,"C22, C24, C26, C27, C28, C31, C32, C33, C34, C39, C40, C42, C43, C46, C48, C49, C50, C51, C53, C68, C69, L108, L151, L61, T02, T03, T04, T05, T07, T08, T09, T10"
1920,Santa Rosa de Viterbo,29,"C24, C26, C27, C28, C31, C32, C33, C34, C39, C40, C42, C43, C48, C49, C50, C51, C53, C68, C69, L151, L61, T02, T03, T04, T05, T07, T08, T09, T10"
1926,Ignacio Zaragoza/Ignacio Allende,28,"C24, C26, C27, C28, C31, C32, C33, C34, C39, C40, C42, C43, C49, C50, C51, C53, C68, L151, L155, L61, T02, T03, T04, T05, T07, T08, T09, T10"
1925,Ignacio Zaragoza/Vicente Guerrero Salesiano,28,"C24, C26, C27, C28, C31, C32, C33, C34, C39, C40, C42, C43, C49, C50, C51, C53, C68, L151, L155, L61, T02, T03, T04, T05, T07, T08, T09, T10"
1919,Vicente Guerrero,27,"C24, C26, C27, C28, C31, C32, C33, C39, C40, C42, C43, C48, C49, C50, C51, C53, C68, L151, L61, T02, T03, T04, T05, T07, T08, T09, T10"
1918,Vergara,23,"C26, C28, C31, C32, C33, C39, C40, C42, C48, C49, C50, C51, C53, C68, L151, L61, T02, T03, T04, T05, T08, T09, T10"
2203,Alameda,21,"C26, C28, C30, C31, C33, C34, C35, C39, C42, C49, C50, C53, C68, L151, L155, T02, T04, T05, T08, T09, T10"
2201,Ignacio Zaragoza/Imss,21,"C23, C28, C39, C40, C42, C46, C48, C50, C68, L108, L151, L152, L155, L61, T02, T03, T04, T05, T08, T09, T10"
2195,Ignacio Zaragoza/Telmex,21,"C21, C22, C23, C28, C39, C40, C42, C46, C48, C50, C68, L108, L151, L152, L61, T02, T03, T04, T05, T08, T10"
1924,Ignacio Zaragoza/Ezequiel Montes,19,"C23, C26, C28, C32, C39, C40, C50, C68, L151, L152, L155, L61, T02, T03, T04, T05, T08, T09, T10"


Isócronas: 2,610 paradas, 7,751 tramos de ruta y 2,693 pares de paradas caminables.


Combinaciones ruta-sentido dibujadas: 218
Copia HTML: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/maps/mapa_tiempos_totales_rutas_gtfs.html
